# Lab01 - Mật mã học cổ điển
**Nhóm 16 - NT101.R11.2**  
**Hà Nguyễn Hiếu An (24520043), Nguyễn Hữu Nghĩa (24521144)**

# Nhiệm vụ 2.4 — Playfair
## Tạo ma trận và chuẩn hóa
1. Chỉ nhận chữ cái ASCII, chuyển thành chữ hoa, gộp `J` vào `I`. Không phiên âm chữ có dấu.
2. Khóa bỏ chữ trùng, giữ thứ tự xuất hiện; bổ sung `ABCDEFGHIKLMNOPQRSTUVWXYZ` để có ma trận 5 × 5.
3. Bản rõ chia cặp: nếu hai chữ giống nhau, chèn `X` sau chữ đầu và xét tiếp chữ thứ hai. Đệm cuối nếu còn một chữ.
4. Khi chữ đang xử lý là `X`, dùng `Q` làm đệm để không tạo cặp `XX`.

## Quy tắc biến đổi từng cặp
| Vị trí | Mã hóa | Giải mã |
|---|---|---|
| Cùng hàng | Dịch phải 1 ô, quay vòng | Dịch trái 1 ô, quay vòng |
| Cùng cột | Dịch xuống 1 ô, quay vòng | Dịch lên 1 ô, quay vòng |
| Khác hàng và cột | Giữ hàng, đổi cột cho nhau | Giữ hàng, đổi cột cho nhau |

Ciphertext được bỏ khoảng trắng/dấu câu như bản rõ, nhưng **không chèn đệm khi giải mã**.
Từ chối ciphertext có số chữ lẻ hoặc cặp trùng vì không phải đầu ra hợp lệ theo quy ước này.
Bản rõ rỗng/không có chữ và khóa rỗng/không có chữ cũng bị từ chối.

**Giới hạn:** giải mã trả bản rõ đã chuẩn hóa, còn ký tự đệm. Không tự xóa `X/Q` vì có thể là chữ thật;
không khôi phục được dấu câu, khoảng trắng, chữ hoa/thường hoặc phân biệt `I/J` ban đầu.

In [1]:
PLAYFAIR_ALPHABET = "ABCDEFGHIKLMNOPQRSTUVWXYZ"


def playfair_normalize(text):
    # Lọc ASCII trước upper(): tránh ký tự như ß biến thành SS.
    return "".join(
        character.upper().replace("J", "I")
        for character in text
        if "A" <= character <= "Z" or "a" <= character <= "z"
    )


def playfair_matrix(key):
    normalized_key = playfair_normalize(key)
    if not normalized_key:
        raise ValueError("Cần nhập text: ")
    unique_letters = "".join(dict.fromkeys(normalized_key + PLAYFAIR_ALPHABET))
    return [list(unique_letters[index:index + 5]) for index in range(0, 25, 5)]


def playfair_show_matrix(key):
    matrix = playfair_matrix(key)
    print("Ma trận Playfair 5 × 5:")
    for row in matrix:
        print(" ".join(row))
    return matrix


def playfair_prepare_plaintext(plaintext):
    text = playfair_normalize(plaintext)
    if not text:
        raise ValueError("Cần nhập text: ")
    pairs = []
    index = 0
    while index < len(text):
        first = text[index]
        padding = "Q" if first == "X" else "X"
        if index + 1 >= len(text) or first == text[index + 1]:
            pairs.append(first + padding)
            index += 1
        else:
            pairs.append(text[index:index + 2])
            index += 2
    return "".join(pairs)


def playfair_transform_pairs(text, key, direction):
    """Hàm nội bộ: nhận chuỗi cặp đã chuẩn bị, direction = +1 hoặc -1."""
    matrix = playfair_matrix(key)
    positions = {
        character: (row, column)
        for row, values in enumerate(matrix)
        for column, character in enumerate(values)
    }
    result = []
    for index in range(0, len(text), 2):
        row_a, column_a = positions[text[index]]
        row_b, column_b = positions[text[index + 1]]
        if row_a == row_b:
            result.extend((matrix[row_a][(column_a + direction) % 5],
                           matrix[row_b][(column_b + direction) % 5]))
        elif column_a == column_b:
            result.extend((matrix[(row_a + direction) % 5][column_a],
                           matrix[(row_b + direction) % 5][column_b]))
        else:
            result.extend((matrix[row_a][column_b], matrix[row_b][column_a]))
    return "".join(result)


def playfair_encrypt(plaintext, key):
    prepared = playfair_prepare_plaintext(plaintext)
    return playfair_transform_pairs(prepared, key, 1)


def playfair_decrypt(ciphertext, key):
    normalized = playfair_normalize(ciphertext)
    if not normalized:
        raise ValueError("Cần nhập text")
    if len(normalized) % 2:
        raise ValueError("Ciphertext Playfair phải có số chữ cái chẵn sau chuẩn hóa.")
    if any(normalized[i] == normalized[i + 1] for i in range(0, len(normalized), 2)):
        raise ValueError("Không hợp lệ")
    return playfair_transform_pairs(normalized, key, -1)

### Giải mã ciphertext của đề
Khóa: `Harry Potter`
Đề https://pastecode.io/s/c9oktc3h.

In [11]:
PLAYFAIR_TASK_KEY = "Harry Potter"
PLAYFAIR_TASK_CIPHERTEXT = 'ARYWYPHCBVEBYGMPNCYGCNTDNCWTMGRMFTQPLEWTMLREFBEBQEBIYGBFLPHVOAEHKDHEUNGQFEROLEWTMLOPHEQGOSBEROQDWTLCMTHBWLNRKXRYLORYYPHCBVEBYRLGYDMKYGGWKLROANDBWGNERMNGYRLGHEWRTRLMBRHMUDGVODVTEGMCHLGWCMTFODNRRYCMZKODDUTDXGEOPOYRMFRMGUKXRYGHABROVTGQMCEHPRPEOTSEGEQLARYWYPOTMGQDOEXGOAUDHGUTULTNEHFTFHPGXGVPHGURBDMEGWKLETCBOTNTFQLTAEHMTUGEOAHEVEROXGVPHGDEWTEWGQIEDLPILERWPMOATNGQKQEAHBMVRFKBRMKLXODXFREBHMNUKXRYKLRMFLWDDNCN'

playfair_task_matrix = playfair_show_matrix(PLAYFAIR_TASK_KEY)
playfair_task_plaintext = playfair_decrypt(PLAYFAIR_TASK_CIPHERTEXT, PLAYFAIR_TASK_KEY)
print("Số chữ ciphertext:", len(PLAYFAIR_TASK_CIPHERTEXT))
print("p = ")
for playfair_offset in range(0, len(playfair_task_plaintext), 80):
    print(playfair_task_plaintext[playfair_offset:playfair_offset + 80])
print("\nMã hóa lại khớp ciphertext:",
      playfair_encrypt(playfair_task_plaintext, PLAYFAIR_TASK_KEY) == PLAYFAIR_TASK_CIPHERTEXT)

Ma trận Playfair 5 × 5:
H A R Y P
O T E B C
D F G I K
L M N Q S
U V W X Z
Số chữ ciphertext: 404
p = 
HARXRYPOTXTERISASERIESOFSEVENFANTASYNOVELSWRITTENBYBRITISHAUTHORIKROWLINGTHENOVE
LSCHRONICLETHELIVESOFAYOUNGWIZARDHARRYPOTXTERANDHISFRIENDSHERMIONEGRANGERANDRONW
EASLEYALLOFWHOMARESTUDENTSATHOGWARTSSCHOOLOFWITCHCRAFTANDWIZARDRYTHEMAINSTORYARC
CONCERNSHARXRYCONFLICTWITHLORDVOLDEMORTADARKWIZARDWHOINTENDSTOBECOMEIMMORTALOVER
THROWTHEWIZARDGOVERNINGBODYKNOWNASTHEMINISTRYOFMAGICANDSUBIUGATEALLWIZARDSANDMUG
GLES

Mã hóa lại khớp ciphertext: True


### Nhập dữ liệu Playfair

In [3]:
PLAYFAIR_INTERACTIVE = False

if PLAYFAIR_INTERACTIVE:
    try:
        playfair_mode = input("Playfair: 1 = mã hóa, 2 = giải mã: ").strip()
        if playfair_mode not in {"1", "2"}:
            raise ValueError("Chế độ phải là 1 hoặc 2.")
        playfair_input_key = input("k = ")
        playfair_show_matrix(playfair_input_key)
        playfair_input_text = input("C = ")
        if playfair_mode == "1":
            print("p = ", playfair_prepare_plaintext(playfair_input_text))
            print("C = ", playfair_encrypt(playfair_input_text, playfair_input_key))
        else:
            print("p = ", playfair_decrypt(playfair_input_text, playfair_input_key))
    except ValueError as error:
        print("Lỗi:", error)
else:
    print("Đổi PLAYFAIR_INTERACTIVE = True để sử dụng.")

Đổi PLAYFAIR_INTERACTIVE = True để sử dụng.


## Kiểm thử Playfair
Ví dụ chuẩn `playfair example` / `Hide the gold in the tree stump` có ciphertext
`BMODZBXDNABEKUDMUIXMMOUVIF`. Kiểm tra riêng ba quy tắc, quay vòng, chuẩn hóa,
chữ lặp/đệm, mọi cặp chữ khác nhau trong bảng, lỗi đầu vào và ciphertext đề.

In [4]:
import unittest

class TestPlayfair(unittest.TestCase):
    def test_key_matrix(self):
        matrix = playfair_matrix("Harry Potter")
        self.assertEqual(["".join(row) for row in matrix],
                         ["HARYP", "OTEBC", "DFGIK", "LMNQS", "UVWXZ"])
        self.assertEqual(len(set("".join("".join(row) for row in matrix))), 25)
        self.assertEqual(playfair_matrix("harry potter!!"), matrix)

    def test_normalization_and_padding(self):
        self.assertEqual(playfair_normalize("Jig, saw! à ß 12"), "IIGSAW")
        for original, expected in (("BALLOON", "BALXLOON"), ("ABC", "ABCX"),
                                   ("XX", "XQXQ"), ("X", "XQ"), ("J", "IX")):
            with self.subTest(original=original):
                self.assertEqual(playfair_prepare_plaintext(original), expected)

    def test_three_rules_and_wrap(self):
        # Khóa A tạo bảng theo alphabet, không có J.
        for plaintext, ciphertext in (("AB", "BC"), ("EA", "AB"),
                                      ("AF", "FL"), ("VA", "AF"), ("AG", "BF")):
            with self.subTest(plaintext=plaintext):
                self.assertEqual(playfair_encrypt(plaintext, "A"), ciphertext)
                self.assertEqual(playfair_decrypt(ciphertext, "A"), plaintext)

    def test_known_example(self):
        key = "playfair example"
        plaintext = "Hide the gold in the tree stump"
        ciphertext = "BMODZBXDNABEKUDMUIXMMOUVIF"
        self.assertEqual(playfair_encrypt(plaintext, key), ciphertext)
        self.assertEqual(playfair_decrypt(ciphertext, key), "HIDETHEGOLDINTHETREXESTUMP")
        self.assertEqual(playfair_decrypt(ciphertext.lower() + " !", key),
                         playfair_prepare_plaintext(plaintext))

    def test_round_trip_all_pairs_and_padding(self):
        key = "Harry Potter"
        for first in PLAYFAIR_ALPHABET:
            for second in PLAYFAIR_ALPHABET:
                text = first + second
                with self.subTest(text=text):
                    ciphertext = playfair_encrypt(text, key)
                    self.assertEqual(playfair_decrypt(ciphertext, key),
                                     playfair_prepare_plaintext(text))
        for text in ("Meet me at noon!", "XXX", "QQQ", "JIGGLE", "XQX", "AXXA"):
            self.assertEqual(playfair_decrypt(playfair_encrypt(text, key), key),
                             playfair_prepare_plaintext(text))

    def test_invalid_inputs(self):
        for key in ("", "123 à"):
            with self.assertRaises(ValueError):
                playfair_matrix(key)
        for text in ("", "123 à"):
            with self.assertRaises(ValueError):
                playfair_encrypt(text, "key")
        for text in ("", "123", "ABC", "AA", "JJ"):
            with self.assertRaises(ValueError):
                playfair_decrypt(text, "key")

    def test_task_ciphertext(self):
        plaintext = playfair_decrypt(PLAYFAIR_TASK_CIPHERTEXT, PLAYFAIR_TASK_KEY)
        self.assertEqual(len(PLAYFAIR_TASK_CIPHERTEXT), 404)
        self.assertTrue(plaintext.startswith("HARXRYPOTXTERISASERIESOFSEVENFANTASYNOVELS"))
        self.assertEqual(playfair_encrypt(plaintext, PLAYFAIR_TASK_KEY), PLAYFAIR_TASK_CIPHERTEXT)


playfair_test_result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestPlayfair)
)
if not playfair_test_result.wasSuccessful():
    raise RuntimeError("Kiểm thử Playfair thất bại.")

test_invalid_inputs (__main__.TestPlayfair.test_invalid_inputs) ... ok
test_key_matrix (__main__.TestPlayfair.test_key_matrix) ... ok
test_known_example (__main__.TestPlayfair.test_known_example) ... ok
test_normalization_and_padding (__main__.TestPlayfair.test_normalization_and_padding) ... ok
test_round_trip_all_pairs_and_padding (__main__.TestPlayfair.test_round_trip_all_pairs_and_padding) ... ok
test_task_ciphertext (__main__.TestPlayfair.test_task_ciphertext) ... ok
test_three_rules_and_wrap (__main__.TestPlayfair.test_three_rules_and_wrap) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.036s

OK


## Tài liệu tham khảo
- Ciphertext đề: https://pastecode.io/s/c9oktc3h.
- Tham khảo thuật toán/thống kê: https://en.wikipedia.org/wiki/Playfair_cipher.